In [1]:
import importlib
import peak_gene_utils
importlib.reload(peak_gene_utils)
from peak_gene_utils import *

from utils import *
import squidpy as sq

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/anndata

## Load data

In [2]:
# Load cCREs as a DataFrame
ccre_bed = pd.read_csv(
    "mm10-cCREs.bed",
    sep="\t",
    header=None,
    names=["chrom", "start", "end", "ccre_id", "accession", "ccre_type"]
)

In [3]:
promoter_ccres = ccre_bed[ccre_bed["ccre_type"]=="PLS"]

In [4]:
atac = sc.read_h5ad("desc_normalized_atac_peaks.h5ad")
expr = sc.read_h5ad("desc_normalized_rna.h5ad")

In [5]:
sp_merfish = sc.read_h5ad("../../h5ad_files/c_sp_ad.h5ad")

In [6]:
sp_merfish_names = sp_merfish.var.index.str.lower().tolist()

In [7]:
data = np.load("cached_data.npz")
idx_atac = data["idx_atac"]
idx_expr = data["idx_expr"]
pdist_ = data["pdist"]

In [8]:
atac_20000 = atac[:,idx_atac]
expr_2000 = expr[:,idx_expr]

In [9]:
g_p_similarity = pdist_[20000:,:20000]

In [10]:
sq.gr.spatial_autocorr(expr_2000, mode="moran")

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  obj[key] = data


In [11]:
sq.gr.spatial_autocorr(atac_20000, mode="moran")

/nfs/home/students/l.reich/mamba/envs/neu/lib/python3.10/site-packages/squidpy/gr/_utils.py:203: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  obj[key] = data


In [12]:
moran_list = expr_2000.uns["moranI"].head(n=250).index.tolist()

In [13]:
peak_moran_list = atac_20000.uns["moranI"].head(n=1000).index.tolist()

## Get correlated gene-peak pairs

In [14]:
# Get only unannotated peaks near genes in moran_list
no_anno_results_by_gene = get_filtered_peak_results_by_gene(
    gene_list=expr_2000.var_names,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    top_n=200,
    max_distance=100000,
    keep_annotated=False
)

Skipping gm11128: Coordinates missing for gene gm11128.
Skipping a330050f15rik: Coordinates missing for gene a330050f15rik.
Skipping rp23-474j16.2: Coordinates missing for gene rp23-474j16.2.
Skipping rp24-136l4.6: Coordinates missing for gene rp24-136l4.6.
Skipping rp23-2d23.2: Coordinates missing for gene rp23-2d23.2.
Skipping pcnxl2: Coordinates missing for gene pcnxl2.
Skipping mir143hg: Coordinates missing for gene mir143hg.
Skipping rp24-416m6.5: Coordinates missing for gene rp24-416m6.5.
Skipping rp24-267c3.3: Coordinates missing for gene rp24-267c3.3.
Skipping rp24-134n2.1: Coordinates missing for gene rp24-134n2.1.
Skipping rp23-407n2.2: Coordinates missing for gene rp23-407n2.2.
Skipping pvrl3: Coordinates missing for gene pvrl3.
Skipping creb5: Coordinates missing for gene creb5.
Skipping rp24-263g24.2: Coordinates missing for gene rp24-263g24.2.
Skipping rp23-44h21.1: Coordinates missing for gene rp23-44h21.1.


In [15]:
# Get only annotated peaks
anno_results_by_gene = get_filtered_peak_results_by_gene(
    gene_list=moran_list,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    top_n=300,
    max_distance=100000,
    keep_annotated=True
)

Skipping mir143hg: Coordinates missing for gene mir143hg.


In [16]:
anno_results_by_peak = get_genes_associated_with_peaks(
    peak_names=peak_moran_list,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    keep_annotated=True,   # or False, or None to skip
    max_distance=100000,
    top_n_per_peak=10
)

KeyboardInterrupt: 

In [ ]:
no_anno_results_by_peak = get_genes_associated_with_peaks(
    peak_names=peak_moran_list,
    expr=expr_2000,
    atac=atac_20000,
    g_p_similarity=g_p_similarity,
    ccre_bed=ccre_bed,
    keep_annotated=False,   # or False, or None to skip
    max_distance=100_000,
    top_n_per_peak=10
)

## Plot pairs (reduced to 10 pairs for now)

In [ ]:
plot_peak_gene_pairs(
    mapping_dict=no_anno_results_by_peak,
    expr_adata=expr_2000,
    atac_adata=atac_20000,
    mode='peak_to_gene',
    n_pairs=10
)

In [ ]:
plot_peak_gene_pairs(
    mapping_dict=anno_results_by_peak,
    expr_adata=expr_2000,
    atac_adata=atac_20000,
    mode='peak_to_gene',
    n_pairs=10
)

In [ ]:
plot_peak_gene_pairs(
    mapping_dict=anno_results_by_gene,
    expr_adata=expr_2000,
    atac_adata=atac_20000,
    mode='gene_to_peak',
    n_pairs=10
)

In [ ]:
plot_peak_gene_pairs(
    mapping_dict=no_anno_results_by_gene,
    expr_adata=expr_2000,
    atac_adata=atac_20000,
    mode='gene_to_peak',
    n_pairs=10
)

## Peak-Peak correlations

### combine annotated Peaks from both mappings

In [ ]:
# Flatten all peak rows into one DataFrame
p_by_g_df = pd.concat(anno_results_by_gene.values())
p_by_g_df = p_by_g_df[["chrom", "start", "end"]].drop_duplicates()

# Step 1: find promoter-overlapping peaks
by_g_promoter_peaks = get_promoter_overlapping_peaks(p_by_g_df, promoter_ccres=promoter_ccres)

In [ ]:
by_p_list = list(anno_results_by_peak.keys())
g_by_p_df = atac_20000.var.loc[by_p_list, ["chrom", "start", "end"]]  # or your peak dataframe

# Get promoter overlaps
by_p_promoter_peaks = get_promoter_overlapping_peaks(g_by_p_df, promoter_ccres)

In [ ]:
union_peaks = list(set(by_g_promoter_peaks) | set(by_p_promoter_peaks))

In [ ]:
len(union_peaks)

In [ ]:
# get peak-peak similarity matrix
p_p_similarity = pdist_[:20000, :20000]

In [ ]:
correlated_peaks = get_correlated_peaks_near_peaks(
    peak_names=union_peaks,
    atac=atac_20000,
    p_p_similarity=p_p_similarity,
    max_distance=50000,
    top_n_per_peak=5
)

In [ ]:
plot_peak_peak_pairs(
    mapping_dict=correlated_peaks,
    atac_adata=atac_20000,
    n_pairs=10,
    size=25
)

In [ ]:
for p in correlated_peaks.keys():
    if p in anno_results_by_peak.keys():
        for g in anno_results_by_peak[p]:
            sq.pl.spatial_scatter(shape=None, adata=expr_2000, color=g)

## Celltype specific analysis

In [ ]:
# Flatten values from *_by_peak (each value is a list of genes)
genes_from_anno_by_peak = {gene for genes in anno_results_by_peak.values() for gene in genes}
genes_from_no_anno_by_peak = {gene for genes in no_anno_results_by_peak.values() for gene in genes}

# Union all gene names from all sources
all_genes = set(no_anno_results_by_gene.keys()) | \
            set(anno_results_by_gene.keys()) | \
            genes_from_anno_by_peak | \
            genes_from_no_anno_by_peak

# Optionally sort
all_genes = sorted(all_genes)

In [ ]:
# optionally: only w/ CRE annotation
# Flatten values from *_by_peak (each value is a list of genes)
genes_from_anno_by_peak = {gene for genes in anno_results_by_peak.values() for gene in genes}

# Union all gene names from all sources
all_genes = set(anno_results_by_gene.keys()) | \
            genes_from_anno_by_peak

# Optionally sort
all_genes = sorted(all_genes)

### find marker genes independently on scRNA SHARE-seq data

In [ ]:
ad_sc = sc.read_h5ad("../../h5ad_files/c_sc_ad.h5ad")

In [ ]:
ad_sc.var_names = ad_sc.var_names.str.lower()

In [ ]:
sc.tl.rank_genes_groups(ad_sc, groupby="celltype")

In [ ]:
# Extract top 100 marker genes per subclass
marker_genes_df = sc.get.rank_genes_groups_df(ad_sc, group=None, key='rank_genes_groups', pval_cutoff=0.05)

In [ ]:
# Keep top 100 genes per subclass (cell type)
top100_per_subclass = marker_genes_df.groupby("group").head(10)

In [ ]:
sc.pl.rank_genes_groups(ad_sc)

In [ ]:
# Convert your gene list to a set for efficient comparison
all_genes_set = set(all_genes)

# Find overlapping genes
overlapping_genes = all_genes_set.intersection(top100_per_subclass['names'])
print(f"Number of overlapping genes: {len(overlapping_genes)}")

In [ ]:
overlapping_genes

In [ ]:
sq.pl.spatial_scatter(shape=None, adata=expr, color=overlapping_genes)

#### looking at rorb

In [ ]:
top100_per_subclass[top100_per_subclass["names"]=="rorb"]

In [ ]:
if "rorb" in no_anno_results_by_gene:
    print("Found in no_anno_results_by_gene")
    display(no_anno_results_by_gene["rorb"])

if "rorb" in anno_results_by_gene:
    print("Found in anno_results_by_gene")
    display(anno_results_by_gene["rorb"])

In [ ]:
get_overlapping_ccres("chr19:18972253-18972752", ccre_bed)

In [ ]:
rorb_corr_peaks = get_correlated_peaks_near_peaks(
    peak_names=["chr19:18972253-18972752"],
    atac=atac_20000,
    p_p_similarity=p_p_similarity,
    max_distance=100000,
    top_n_per_peak=5
)

In [ ]:
rorb_corr_peaks

In [ ]:
get_overlapping_ccres('chr19:18967342-18967841', ccre_bed)

In [ ]:
plot_peak_peak_pairs({"chr19:18972253-18972752": ["chr19:18967342-18967841"]}, atac_adata=atac_20000)

In [ ]:
plot_peak_accessibility("chr19:18972253-18972752", atac_20000)

In [ ]:
plot_peak_accessibility("chr19:18967342-18967841", atac_20000)

#### looking for promoter genes

In [ ]:
marker_genes = set(top100_per_subclass['names'].str.lower())

In [ ]:
def find_matching_peaks(peak_to_genes_dict, union_peaks, marker_genes):
    return {
        peak for peak, genes in peak_to_genes_dict.items()
        if peak in union_peaks and any(g.lower() in marker_genes for g in genes)
    }

In [ ]:
matching_peaks_anno = find_matching_peaks(anno_results_by_peak, union_peaks, marker_genes)
matching_peaks_no_anno = find_matching_peaks(no_anno_results_by_peak, union_peaks, marker_genes)

# Combine if you want the full set
matching_promoter_peaks = matching_peaks_anno | matching_peaks_no_anno

In [ ]:
matching_promoter_peaks

##### get no_anno peaks

In [ ]:
def find_matching_peaks_from_gene(gene_to_result_dict, union_peaks, marker_genes):
    matching_peaks = set()
    for gene, df in gene_to_result_dict.items():
        gene_lc = gene.lower()
        if gene_lc in marker_genes:
            peaks = df.index if isinstance(df, pd.DataFrame) else []
            matching_peaks.update(p for p in peaks if p in union_peaks)
    return matching_peaks

In [ ]:
matching_peaks_anno_gene = find_matching_peaks_from_gene(anno_results_by_gene, union_peaks, marker_genes)
matching_peaks_no_anno_gene = find_matching_peaks_from_gene(no_anno_results_by_gene, union_peaks, marker_genes)

In [ ]:
all_matching_peaks = (
    matching_peaks_anno |
    matching_peaks_no_anno |
    matching_peaks_anno_gene |
    matching_peaks_no_anno_gene
)

In [ ]:
all_matching_peaks

In [ ]:
get_overlapping_ccres('chr19:5726044-5726543', ccre_bed)

In [ ]:
get_overlapping_ccres('chr2:128698733-128699232', ccre_bed)